# Block 1 - Colab setup + baseline (InstantSplat on Mip-NeRF 360)
**Project:** What Makes AI-Based Sparse-View 3D Gaussian Splatting Work? - Farida Gaber & Mona Yasser

**What this notebook does (tracker tasks F1.1 - F3.2):**

| Step | What | Why |
|---|---|---|
| 1 | Check GPU, connect Google Drive | Colab erases everything; Drive keeps our files |
| 2 | Install InstantSplat + compile the GPU code | Nothing runs without it |
| 3 | (optional) Smoke test on the sample scene | Quick proof the install works |
| 4 | Download Mip-NeRF 360 (8 scenes) | The data for all experiments |
| 5 | Set the train/test split rule | We follow Free360: every 8th photo = test |
| 6 | Define the "run one experiment" function | The seed of our later batch script |
| 7 | Run the baseline (full InstantSplat) on 2 scenes x 3/6/9 photos | Check our setup against the published numbers |
| 8 | Results table + text for LOG.md | Evidence for milestone M1 |

**How to use**
1. `Runtime -> Change runtime type -> T4 GPU`
2. `Runtime -> Run all`
3. If the session disconnects: run all again. **Finished work is skipped.**

**Time:** about 1.5-2 hours (setup ~10 min, data ~5 min from the Drive copy, then 6 runs of ~9-21 min each on a T4).
**Manual step (not in this notebook):** share the Drive folder `3dgs-project` with Mona (Editor).

**Version 2 (9 Nov) - changes after the first full run:**
- The test-pose start is aligned to the reference poses (InstantSplat's default start was ~80-90 degrees off).
- Pose error is now measured by **our own `pose_error.py`** (InstantSplat's ATE/RPE mix two pose formats and are not reliable). Runs finished with version 1 have no valid pose error, so they run again once.
- The dataset is copied to Drive once (~small), so later sessions do not download 12 GB again.
- The camera poses of every run (before and after refinement) are saved to Drive.

### Settings (change only if you know why)
All other cells read these values.

In [ ]:
DRIVE = "/content/drive/MyDrive/3dgs-project"   # everything we want to keep goes here
REPO = "/content/InstantSplat"                  # code (re-created every session)
DATA = "/content/data"                          # dataset (re-downloaded every session)
LOCAL_OUT = "/content/out"                      # heavy model files (not kept)

RES_DIR = "images_8"            # image size used for ALL experiments (smallest = fastest on a T4)
ALL_SCENES = ["bicycle", "garden", "stump", "treehill", "bonsai", "counter", "kitchen", "room"]  # flowers excluded (as Free360)
SCENES_BASELINE = ["garden", "bonsai"]          # this notebook runs the baseline on these
VIEWS = [3, 6, 9]                               # number of training photos
ITERS = 1000                                    # training iterations (InstantSplat default for eval)
TEST_STRIDE = 8                                 # every 8th photo is a test photo (Free360 / 3DGS rule)
OPTIM_TEST_POSE_ITER = 500                      # test-photo pose optimization steps (InstantSplat default)
TAG = "C3_baseline"                             # name of this configuration (C3 = full InstantSplat)
RUN_SMOKE_TEST = False                          # already proven once; set True to repeat
FORCE = False                                   # True = redo runs that are already finished

## Step 1 - GPU and Google Drive
**What we're doing:** checking that Colab gave us a GPU, and connecting Google Drive.
**Why it matters:** InstantSplat needs an NVIDIA GPU. Drive is where setup files and results are saved, because Colab deletes everything when the session ends.

In [ ]:
import subprocess, os, sys, json, time, re, shutil, struct, threading, datetime
from pathlib import Path
import numpy as np

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit("NO GPU found. Use: Runtime -> Change runtime type -> T4 GPU, then Run all again.")
GPU_NAME = gpu.stdout.strip()
print("GPU:", GPU_NAME)
if "T4" not in GPU_NAME:
    print("WARNING: this is not a T4. All FINAL results must come from the same GPU type - tell your teammate.")

from google.colab import drive
drive.mount("/content/drive")
for d in ["", "checkpoints", "wheels", "results", "logs", "data", "scripts"]:
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)


def sh(cmd, cwd=None, check=True, quiet=False):
    # Run a shell command, show its output live, raise a clear error if it fails.
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if not quiet:
            print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}\n" + "".join(lines[-30:]))
    return "".join(lines)


print("OK - Drive folder ready:", DRIVE)

## Step 2 - Install InstantSplat
**What we're doing:** downloading InstantSplat and the MASt3R AI model, installing the libraries, and compiling 3 small GPU programs (they draw the 3D "blobs"). It also applies the 4 fixes we found:
1. PyTorch 2.6+ refuses to open the MASt3R file unless we allow it (the file is the official one from Naver).
2. The renderer needs one missing `#include <cstdint>` line for the new compiler.
3. `open3d` must be installed (InstantSplat imports it).
4. The pose-error code uses an old NumPy name (`np.linalg.linalg`) that new NumPy removed.

**Why it matters:** this is the step most likely to break when Colab updates its software. The compiled programs are saved on Drive, so next time the install takes minutes, not 20 minutes.

In [ ]:
SETUP_SH = f"{DRIVE}/setup_colab.sh"
Path(SETUP_SH).write_text(r'''#!/bin/bash
set -e
DRIVE=/content/drive/MyDrive/3dgs-project
cd /content
[ -d InstantSplat ] || git clone -q --recursive https://github.com/NVlabs/InstantSplat.git
cd InstantSplat
# pin the exact InstantSplat version used for every run of the study
if [ "$(git rev-parse --short HEAD)" != "b951567" ]; then
  git fetch -q origin && git checkout -q b951567 && git submodule update -q --init --recursive
fi
[ "$(git rev-parse --short HEAD)" = "b951567" ] || { echo "InstantSplat is not at b951567"; exit 1; }
mkdir -p mast3r/checkpoints
CK=MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth
if [ ! -f $DRIVE/checkpoints/$CK ]; then
  wget -q -O $DRIVE/checkpoints/$CK.part https://download.europe.naverlabs.com/ComputerVision/MASt3R/$CK
  mv $DRIVE/checkpoints/$CK.part $DRIVE/checkpoints/$CK
fi
[ -f mast3r/checkpoints/$CK ] || cp $DRIVE/checkpoints/$CK mast3r/checkpoints/
sed -i "s/torch.load(model_path, map_location='cpu')/torch.load(model_path, map_location='cpu', weights_only=False)/" mast3r/model.py
for f in submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer.h; do
  grep -q "<cstdint>" $f || sed -i '1i #include <cstdint>' $f
done
grep -rl "np.linalg.linalg" --include=*.py . | xargs -r sed -i 's/np\.linalg\.linalg\./np.linalg./g'
grep -v gradio requirements.txt > /content/req.txt
pip install -q -r /content/req.txt
W=$DRIVE/wheels
export TORCH_CUDA_ARCH_LIST=7.5
CHECK='import diff_gaussian_rasterization, simple_knn, fused_ssim'
if ls $W/*.whl >/dev/null 2>&1 && pip install -q $W/*.whl && python -c "$CHECK"; then
  echo "cached wheels OK"
else
  echo "building the GPU programs (first time, or Colab changed its software) ..."
  rm -f $W/*.whl; mkdir -p $W
  pip wheel -q --no-build-isolation --no-deps -w $W submodules/simple-knn submodules/diff-gaussian-rasterization submodules/fused-ssim
  pip install -q --force-reinstall --no-deps $W/*.whl
fi
python -c "import torch; $CHECK; print('setup OK', torch.__version__)"
''')

quick = subprocess.run(["python", "-c", "import diff_gaussian_rasterization, simple_knn, fused_ssim, open3d"],
                       capture_output=True, text=True)
if os.path.isdir(REPO) and quick.returncode == 0:
    print("Already installed in this session - skipping.")
else:
    sh(f"bash {SETUP_SH}")

COMMIT = sh(f"cd {REPO} && git rev-parse --short HEAD", quiet=True).strip()
os.makedirs(f"{DRIVE}/env", exist_ok=True)
ENV_FILE = f"{DRIVE}/env/pip_freeze_{datetime.datetime.now():%Y-%m-%d_%H%M}.txt"   # one file per session
sh(f"pip freeze > '{ENV_FILE}'", quiet=True)
print("OK - InstantSplat commit:", COMMIT, "| package list saved to Drive:", ENV_FILE)

## Step 3 - Smoke test (optional)
**What we're doing:** running InstantSplat on its own sample scene (Santorini, 3 photos).
**Why it matters:** a fast proof that the install works. We already did this once, so it is **off** by default. Set `RUN_SMOKE_TEST = True` in the settings to repeat it (about 8 minutes).

In [ ]:
if RUN_SMOKE_TEST:
    S, M = "assets/sora/Santorini/", "output_test/Santorini/3_views"
    sh(f"rm -rf {M}; mkdir -p {M}", cwd=REPO)
    sh(f"python -W ignore init_geo.py -s {S} -m {M} --n_views 3 --focal_avg --co_vis_dsp --conf_aware_ranking", cwd=REPO, quiet=True)
    sh(f"python train.py -s {S} -m {M} -r 1 --n_views 3 --iterations 1000 --pp_optimizer --optim_pose", cwd=REPO, quiet=True)
    print(sh(f"ls {M}", cwd=REPO))
    print("OK - smoke test passed")
else:
    print("Skipped (RUN_SMOKE_TEST = False)")

## Step 4 - Download the dataset
**What we're doing:** getting Mip-NeRF 360 and keeping only what we need for our 8 scenes: the small photos (`images_8`) and the reference camera poses (`sparse`). The first time, it downloads the full zips (~12 GB) and saves a small copy to Drive (`data/`); later sessions just unpack that copy. Then we add a link called `images` that points to `images_8`, because InstantSplat looks for a folder with that name.
**Why it matters:** this is the data for every experiment. The smallest image size keeps each run fast on a T4. The reference poses are what we compare the AI's poses with.

In [ ]:
URL = "https://storage.googleapis.com/gresearch/refraw360"
MAIN = ["bicycle", "bonsai", "counter", "garden", "kitchen", "room", "stump"]   # inside 360_v2.zip
EXTRA = ["treehill"]                                                              # inside 360_extra_scenes.zip
CACHE_TAR = f"{DRIVE}/data/mipnerf360_8scenes_{RES_DIR}.tar"   # small copy on Drive (photos + poses only)
os.makedirs(DATA, exist_ok=True)
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"Free disk: {free_gb:.0f} GB")


def ready(s):
    return Path(f"{DATA}/{s}/{RES_DIR}").is_dir() and Path(f"{DATA}/{s}/sparse/0/images.bin").exists()


if not all(ready(s) for s in ALL_SCENES) and os.path.exists(CACHE_TAR):
    print("Unpacking the dataset copy from Drive ...")
    sh(f"tar -xf '{CACHE_TAR}' -C {DATA}")
if not all(ready(s) for s in ALL_SCENES):
    assert free_gb > 40, "Not enough free disk on Colab (need > 40 GB) to download the dataset."


def fetch(zipname, scenes):
    todo = [s for s in scenes if s in ALL_SCENES and not ready(s)]
    if not todo:
        return
    if not Path(f"{DATA}/{zipname}").exists():
        print("Downloading", zipname, "(this takes a few minutes) ...")
        sh(f"wget -c -nv {URL}/{zipname} -O {DATA}/{zipname}")
    members = " ".join(f'"{s}/{RES_DIR}/*" "{s}/sparse/*"' for s in todo)
    sh(f"cd {DATA} && unzip -q -o {zipname} {members}")


fetch("360_v2.zip", MAIN)
fetch("360_extra_scenes.zip", EXTRA)

SCENE_COUNTS = {}
for s in ALL_SCENES:
    if not ready(s):
        raise RuntimeError(f"Scene '{s}' is missing after extraction. Check the zip layout "
                           f"(unzip -l {DATA}/360_extra_scenes.zip | head).")
    link = Path(f"{DATA}/{s}/images")
    if not link.exists():
        os.symlink(RES_DIR, link)
    SCENE_COUNTS[s] = len(os.listdir(f"{DATA}/{s}/{RES_DIR}"))

if not os.path.exists(CACHE_TAR):
    members = " ".join(f"{s}/{RES_DIR} {s}/sparse" for s in ALL_SCENES)
    sh(f"cd {DATA} && tar -cf /content/cache.tar {members} && cp /content/cache.tar '{CACHE_TAR}.part' "
       f"&& mv '{CACHE_TAR}.part' '{CACHE_TAR}' && rm /content/cache.tar")
    for z in ["360_v2.zip", "360_extra_scenes.zip"]:          # free the Colab disk
        Path(f"{DATA}/{z}").unlink(missing_ok=True)
print(f"Dataset copy on Drive: {os.path.getsize(CACHE_TAR) / 1e6:.0f} MB")
print("OK - photos per scene:", SCENE_COUNTS)

## Step 5 - Train/test split rule
**What we're doing:** changing one function in InstantSplat (`split_train_test`). InstantSplat normally uses 12 test photos chosen by its own rule. We follow **Free360** instead:
- **test photos** = every 8th photo of the scene (the standard 3DGS rule),
- **training photos** = 3, 6 or 9 photos spaced evenly among the remaining photos.

We also save the exact list of photos for every scene to Drive (`views_manifest.json`) and check that the photo order matches the order of the reference poses.
**Why it matters:** the same photos must be used in every experiment, and the rule must match the paper we compare with. The same function is used for the AI step and for the pose-error step, so they stay consistent.

In [ ]:
SPLIT_SRC = '''def split_train_test(image_files, llffhold=8, n_views=None, verbose=True):
    # FREE360-STYLE SPLIT (patched by our notebook):
    # test = every llffhold-th image (idx % llffhold == 0); train = n_views images spaced evenly among the rest
    n = len(image_files)
    test_idx = [i for i in range(n) if i % llffhold == 0]
    train_pool = [i for i in range(n) if i % llffhold != 0]
    sparse_idx = np.linspace(0, len(train_pool) - 1, num=n_views, dtype=int)
    train_idx = [train_pool[i] for i in sparse_idx]
    if verbose:
        print(">> Spliting Train-Test Set (Free360-style): ")
        print(" - train_set_indices:  ", train_idx)
        print(" - test_set_indices:   ", test_idx)
    train_img_files = [image_files[i] for i in train_idx]
    test_img_files = [image_files[i] for i in test_idx]
    return train_img_files, test_img_files
'''

SFM = Path(f"{REPO}/utils/sfm_utils.py")
text = SFM.read_text()
if "FREE360-STYLE SPLIT" in text:
    print("Split rule already patched.")
else:
    new_text, n_sub = re.subn(r"def split_train_test\(.*?\n(?=def get_sorted_image_files)", lambda m: SPLIT_SRC + "\n\n",
                              text, flags=re.S)
    assert n_sub == 1, "Could not find split_train_test in sfm_utils.py (InstantSplat changed?)."
    compile(new_text, "sfm_utils.py", "exec")          # syntax check before saving
    SFM.write_text(new_text)
    print("Split rule patched in", SFM)

# --- exact photo lists (same function as the patch) ---
exec(SPLIT_SRC)


def colmap_image_names(path):
    # read only the image names from a COLMAP images.bin file
    names = []
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        for _ in range(n):
            f.read(64)                      # id, rotation, translation, camera id
            name = b""
            while True:
                c = f.read(1)
                if c == b"\x00":
                    break
                name += c
            npts = struct.unpack("<Q", f.read(8))[0]
            f.seek(24 * npts, 1)            # skip 2D points
            names.append(name.decode())
    return names


def instantsplat_order(folder):
    # same order InstantSplat uses: by the number inside the file name
    files = [p for p in Path(folder).iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}]
    def num(p):
        m = re.search(r"\d+", p.stem)
        return int(m.group()) if m else 10**18
    return [p.name for p in sorted(files, key=num)]


manifest = {}
for s in ALL_SCENES:
    ours = instantsplat_order(f"{DATA}/{s}/{RES_DIR}")
    colmap = sorted(colmap_image_names(f"{DATA}/{s}/sparse/0/images.bin"))   # the pose reader sorts by name
    assert ours == colmap, (f"{s}: photo order differs between InstantSplat ({len(ours)}) and the COLMAP poses "
                            f"({len(colmap)}). Tell Farida - the split would be wrong.")
    test = split_train_test(ours, TEST_STRIDE, 3, verbose=False)[1]
    manifest[s] = {"n_images": len(ours), "test_images": test,
                   "train_images": {str(v): split_train_test(ours, TEST_STRIDE, v, verbose=False)[0] for v in VIEWS}}
json.dump(manifest, open(f"{DRIVE}/views_manifest.json", "w"), indent=1)
print("OK - photo order verified for all scenes; lists saved to Drive/views_manifest.json")
print({s: (m["n_images"], len(m["test_images"])) for s, m in manifest.items()}, "<- (photos, test photos)")

# --- Patch 2: start each test photo from its reference pose, aligned to the AI's training poses ---
# (InstantSplat's default interpolation started test poses ~80-90 degrees off in 360-degree scenes; checked on garden)
INIT_FN = '''
def aligned_test_init(source_path, train_files, test_files, train_w2c):
    # ALIGNED TEST-POSE INIT: Sim(3) (Umeyama) from reference to estimated training camera centres,
    # then the reference test poses are mapped into the estimated frame. Same rule for every configuration.
    import numpy as np, os
    from scene.colmap_loader import read_extrinsics_binary, qvec2rotmat
    gt = {os.path.basename(v.name): v for v in read_extrinsics_binary(os.path.join(source_path, "sparse/0/images.bin")).values()}
    def gt_w2c(f):
        e = gt[os.path.basename(f)]
        M = np.eye(4); M[:3, :3] = qvec2rotmat(e.qvec); M[:3, 3] = e.tvec
        return M
    est_c2w = np.linalg.inv(np.asarray(train_w2c))
    gt_c2w = np.stack([np.linalg.inv(gt_w2c(f)) for f in train_files])
    X, Y = gt_c2w[:, :3, 3], est_c2w[:, :3, 3]
    mx, my = X.mean(0), Y.mean(0); Xc, Yc = X - mx, Y - my
    U, D, Vt = np.linalg.svd(Yc.T @ Xc / len(X)); S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[2, 2] = -1
    R = U @ S @ Vt; s = np.trace(np.diag(D) @ S) / Xc.var(0).sum(); t = my - s * R @ mx
    out = []
    for f in test_files:
        g = np.linalg.inv(gt_w2c(f)); c2w = np.eye(4)
        c2w[:3, :3] = R @ g[:3, :3]; c2w[:3, 3] = s * R @ g[:3, 3] + t
        out.append(np.linalg.inv(c2w))
    return np.stack(out)
'''
IG = Path(f"{REPO}/init_geo.py")
ig = IG.read_text()
if "ALIGNED TEST-POSE INIT" in ig:
    print("Test-pose start already patched.")
else:
    anchor = "        save_extrinsic(sparse_1_path, pose_test_init, test_img_files, image_suffix)"
    assert ig.count(anchor) == 1 and ig.count("\ndef main(") == 1, "init_geo.py changed - tell Farida."
    ig = ig.replace(anchor, "        pose_test_init = aligned_test_init(source_path, train_img_files, test_img_files, extrinsics_w2c)\n" + anchor)
    ig = ig.replace("\ndef main(", INIT_FN + "\n\ndef main(", 1)
    compile(ig, "init_geo.py", "exec")
    IG.write_text(ig)
    print("Test-pose start patched in", IG)

## Step 6 - The "run one experiment" function
**What we're doing:** defining `run_one(scene, n_views)`. It runs the 4 InstantSplat stages and saves a small result file:
1. **init_geo** - the AI (MASt3R) predicts the camera poses and a 3D point cloud from the training photos,
2. **train** - the 3D blobs are trained and the poses are corrected during training,
3. **render test** - draws the test photos (the test poses are tuned first),
4. **metrics** - PSNR, SSIM, LPIPS (image quality), then **our `pose_error.py`** compares the training camera poses with the reference poses, both before refinement (straight from the AI) and after refinement:
   - `ate` = camera-position error after one scale+rotation+shift alignment, divided by the scene size (0 = perfect, 0.1 = off by 10% of the scene size),
   - `rot_deg` = camera-direction error in degrees, `rpe_t` / `rpe_deg` = error of the relative motion between every pair of cameras.

It also records the time of each stage and the peak GPU memory, and saves the camera poses to Drive. Finished runs are skipped, so you can restart safely.
**Why it matters:** this function is the base of our later batch script (96 runs).

In [ ]:
POSE_ERROR_SRC = '"""Camera-pose error for one sparse-view run (tracker task F2.3).\n\nWhy our own script: InstantSplat\'s metrics.py builds the reference pose from a camera-to-world\nrotation and a world-to-camera translation (mixed), and compares it with its own world-to-camera\nposes. Its ATE / RPE numbers are therefore not reliable (RPE_r came out at 78-145 degrees).\n\nHere both sides are turned into camera centres + camera orientations, aligned with ONE Sim(3)\n(Umeyama: scale, rotation, shift) fitted on the training camera centres (plus short points along each\ncamera axis, so the fit stays stable when 3 cameras are nearly on a line - version 2), then:\n  ate      RMSE of the camera-centre error, divided by the scene size\n           (mean distance of the reference centres from their centroid) -> unit-free\n  rot_deg  mean angle between aligned and reference camera orientations (degrees)\n  rpe_t    relative-pose translation error over ALL camera pairs, divided by the scene size\n  rpe_deg  relative-pose rotation error over ALL camera pairs (degrees; needs no alignment)\nSparse views are not a video, so all pairs are used instead of "consecutive" frames.\n\nUsage:\n  python pose_error.py --images-bin <scene>/sparse/0/images.bin \\\n                       --est <model>/pose/ours_1000/pose_optimized.npy \\\n                       --names IMG_1.JPG,IMG_9.JPG,...      (training photos, same order as --est)\n  python pose_error.py --selftest\n"""\nimport argparse\nimport json\nimport struct\n\nimport numpy as np\n\n\n# ----------------------------------------------------------------- reading\ndef read_images_bin(path):\n    """COLMAP images.bin -> {name: (R_w2c 3x3, t_w2c 3)}."""\n    out = {}\n    with open(path, "rb") as f:\n        (n,) = struct.unpack("<Q", f.read(8))\n        for _ in range(n):\n            f.read(4)                                         # image_id\n            qw, qx, qy, qz, tx, ty, tz = struct.unpack("<7d", f.read(56))\n            f.read(4)                                         # camera_id\n            name = b""\n            while (c := f.read(1)) != b"\\x00":\n                name += c\n            (n_pts,) = struct.unpack("<Q", f.read(8))\n            f.read(24 * n_pts)                                # x, y, point3D_id per point\n            out[name.decode()] = (qvec_to_R(np.array([qw, qx, qy, qz])), np.array([tx, ty, tz]))\n    return out\n\n\ndef qvec_to_R(q):\n    w, x, y, z = q / np.linalg.norm(q)\n    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],\n                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],\n                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])\n\n\ndef w2c_to_cam(R_w2c, t_w2c):\n    """world-to-camera -> (camera-to-world rotation, camera centre)."""\n    return R_w2c.T, -R_w2c.T @ t_w2c\n\n\n# ----------------------------------------------------------------- maths\ndef umeyama(X, Y):\n    """Sim(3) with Y ~ s * R @ X + t (X, Y: N x 3)."""\n    mx, my = X.mean(0), Y.mean(0)\n    Xc, Yc = X - mx, Y - my\n    U, D, Vt = np.linalg.svd(Yc.T @ Xc / len(X))\n    S = np.eye(3)\n    if np.linalg.det(U) * np.linalg.det(Vt) < 0:\n        S[2, 2] = -1\n    R = U @ S @ Vt\n    s = np.trace(np.diag(D) @ S) / (Xc ** 2).sum(1).mean()\n    return s, R, my - s * R @ mx\n\n\ndef angle_deg(Ra, Rb):\n    """Rotation angle between Ra and Rb, via atan2 (accurate also for very small angles)."""\n    M = Ra.T @ Rb\n    c = (np.trace(M) - 1) / 2\n    s = np.linalg.norm([M[2, 1] - M[1, 2], M[0, 2] - M[2, 0], M[1, 0] - M[0, 1]]) / 2\n    return float(np.degrees(np.arctan2(s, c)))\n\n\nVERSION = 2   # v2 (12 Nov): alignment uses camera centres AND camera axes (stable for 3 cameras)\n\n\nAXIS_WEIGHT = 0.1   # tested: removes the near-line artifact (7.4 -> 0.5 deg) and keeps single-camera errors honest\n\n\ndef frame_points(rot, ctr):\n    """Camera centres plus one short point along each camera axis (0.1 x the frame\'s size).\n    The axis points make the Sim(3) fit well-posed when the centres are nearly on a line (with 3\n    cameras, centres alone leave the rotation about that line almost free), while the centres still\n    dominate the fit, so one camera\'s orientation error barely spreads to the others."""\n    size = np.linalg.norm(ctr - ctr.mean(0), axis=1).mean()\n    pts = [ctr]\n    for k in range(3):\n        pts.append(ctr + AXIS_WEIGHT * size * rot[:, :, k])\n    return np.concatenate(pts, 0)\n\n\ndef flatness(ctr):\n    """Second / first singular value of the centred camera positions: 0 = cameras on a line."""\n    sv = np.linalg.svd(ctr - ctr.mean(0), compute_uv=False)\n    return float(sv[1] / sv[0]) if sv[0] > 0 else 0.0\n\n\ndef pose_error(ref_rot, ref_ctr, est_rot, est_ctr):\n    """All inputs: camera-to-world rotations (N,3,3) and camera centres (N,3), same camera order.\n    ate / rot_deg / rpe_t depend on the Sim(3) alignment; rpe_deg does not need any alignment.\n    ate is only comparable between runs with the SAME number of cameras (the fit absorbs more of the\n    error when there are fewer cameras)."""\n    ref_rot, ref_ctr, est_rot, est_ctr = map(np.asarray, (ref_rot, ref_ctr, est_rot, est_ctr))\n    n = len(ref_ctr)\n    assert n == len(est_ctr) >= 3, "need the same number of cameras on both sides (at least 3)"\n    s, R, t = umeyama(frame_points(est_rot, est_ctr), frame_points(ref_rot, ref_ctr))\n    al_ctr = (s * (R @ est_ctr.T)).T + t\n    al_rot = R @ est_rot\n    size = np.linalg.norm(ref_ctr - ref_ctr.mean(0), axis=1).mean()\n\n    ate = np.sqrt(((al_ctr - ref_ctr) ** 2).sum(1).mean()) / size\n    rot = np.mean([angle_deg(ref_rot[i], al_rot[i]) for i in range(n)])\n    rpe_t, rpe_r = [], []\n    for i in range(n):\n        for j in range(i + 1, n):\n            # relative motion of camera j seen from camera i\n            g_t = ref_rot[i].T @ (ref_ctr[j] - ref_ctr[i])\n            e_t = al_rot[i].T @ (al_ctr[j] - al_ctr[i])\n            rpe_t.append(np.linalg.norm(g_t - e_t) / size)\n            rpe_r.append(angle_deg(ref_rot[i].T @ ref_rot[j], al_rot[i].T @ al_rot[j]))\n    return {"ate": float(ate), "rot_deg": float(rot), "rpe_t": float(np.mean(rpe_t)),\n            "rpe_deg": float(np.mean(rpe_r)), "n_cams": n, "sim3_scale": float(s),\n            "flatness": flatness(ref_ctr), "pose_error_version": VERSION}\n\n\ndef evaluate_run(images_bin, est_npy, names):\n    """est_npy: InstantSplat pose_optimized.npy / pose_org.npy = world-to-camera 4x4, training order."""\n    ref = read_images_bin(images_bin)\n    est = np.load(est_npy).reshape(-1, 4, 4)\n    assert len(est) == len(names), f"{len(est)} estimated poses but {len(names)} names"\n    ref_cam = [w2c_to_cam(*ref[nm]) for nm in names]\n    est_cam = [w2c_to_cam(P[:3, :3], P[:3, 3]) for P in est]\n    return pose_error([r for r, _ in ref_cam], [c for _, c in ref_cam],\n                      [r for r, _ in est_cam], [c for _, c in est_cam])\n\n\n# ----------------------------------------------------------------- self-test\ndef random_rot(rng):\n    q = rng.normal(size=4)\n    return qvec_to_R(q)\n\n\ndef selftest():\n    rng = np.random.default_rng(0)\n    n = 9\n    rot = np.stack([random_rot(rng) for _ in range(n)])\n    ctr = rng.normal(size=(n, 3)) * 3\n\n    # 1) identity: estimate == reference\n    e = pose_error(rot, ctr, rot, ctr)\n    assert all(e[k] < 1e-6 for k in ("ate", "rot_deg", "rpe_t", "rpe_deg")), e\n\n    # 2) same cameras in another frame (scale 0.3, rotated, shifted) -> still ~0\n    s, Rg, tg = 0.3, random_rot(rng), rng.normal(size=3)\n    e = pose_error(rot, ctr, Rg @ rot, (s * (Rg @ ctr.T)).T + tg)\n    assert all(e[k] < 1e-6 for k in ("ate", "rot_deg", "rpe_t", "rpe_deg")), e\n\n    # 3) world-to-camera round trip (the format InstantSplat saves)\n    w2c = [(r.T, -r.T @ c) for r, c in zip(rot, ctr)]\n    back = [w2c_to_cam(*p) for p in w2c]\n    assert np.allclose([b[1] for b in back], ctr) and np.allclose([b[0] for b in back], rot)\n\n    # 4) a known 10-degree error on one camera is detected\n    bad = rot.copy()\n    ax = np.array([0.0, 0.0, np.radians(10)])\n    K = np.array([[0, -ax[2], ax[1]], [ax[2], 0, -ax[0]], [-ax[1], ax[0], 0]])\n    bad[0] = rot[0] @ (np.eye(3) + np.sin(np.radians(10)) / np.radians(10) * K\n                       + (1 - np.cos(np.radians(10))) / np.radians(10) ** 2 * K @ K)\n    e = pose_error(rot, ctr, bad, ctr)\n    # the axis points spread a tiny part of one camera\'s error over the others; the mean stays ~10/n\n    assert abs(e["rot_deg"] - 10 / n) < 0.05 * 10 / n and e["rpe_deg"] > 1.0, e\n\n    # 5) 3 cameras almost on a line, PERFECT rotations, 0.5 % position noise: the reported rotation\n    #    error must stay small (centre-only alignment gave a median of ~7.6 deg here)\n    def cam(a, r=3.0):\n        c = np.array([np.cos(a), np.sin(a), 0.2]) * r\n        z = -c / np.linalg.norm(c); x = np.cross([0, 0, 1], z); x /= np.linalg.norm(x); y = np.cross(z, x)\n        return np.stack([x, y, z], 1), c\n    errs = []\n    for _ in range(200):\n        R3, C3 = map(np.array, zip(*[cam(np.radians(a)) for a in (0, 180, 358)]))\n        errs.append(pose_error(R3, C3, R3, C3 + rng.normal(scale=0.015, size=C3.shape))["rot_deg"])\n    assert np.median(errs) < 1.0, np.median(errs)\n    print("selftest OK - identity, Sim(3)-invariance, w2c round trip, a known 10-degree error and "\n          "near-collinear 3-camera stability all pass")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--images-bin")\n    ap.add_argument("--est")\n    ap.add_argument("--names", help="comma-separated training photo names, in the order of --est")\n    ap.add_argument("--selftest", action="store_true")\n    a = ap.parse_args()\n    if a.selftest:\n        selftest()\n    else:\n        print(json.dumps(evaluate_run(a.images_bin, a.est, a.names.split(",")), indent=1))\n'
# our pose-error script (same file as scripts/pose_error.py in the GitHub repo)
Path(f"{DRIVE}/scripts/pose_error.py").write_text(POSE_ERROR_SRC)
Path("/content/pose_error.py").write_text(POSE_ERROR_SRC)
sys.path.insert(0, "/content")
import importlib, pose_error
importlib.reload(pose_error)
pose_error.selftest()
POSE_KEYS = ["ate", "rot_deg", "rpe_t", "rpe_deg"]


class VramSampler(threading.Thread):
    # records the highest GPU memory used while a run is going
    def __init__(self):
        super().__init__(daemon=True)
        self.peak, self.stop_flag = 0, False

    def run(self):
        while not self.stop_flag:
            r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                               capture_output=True, text=True)
            try:
                self.peak = max(self.peak, int(r.stdout.split()[0]))
            except Exception:
                pass
            time.sleep(2)


def run_one(scene, n_views, tag=TAG, force=FORCE):
    run_name = f"{tag}/{scene}/{n_views}_views"
    model = f"{LOCAL_OUT}/{run_name}"
    drive_dir = f"{DRIVE}/results/{run_name}"
    result_file = f"{drive_dir}/result.json"
    if os.path.exists(result_file) and not force:
        old = json.load(open(result_file))
        if old.get("pose_metric") == "pose_error.py":
            print(f"skip (already done): {run_name}")
            return old
        print(f"redo (finished with notebook v1, no valid pose error): {run_name}")

    src = f"{DATA}/{scene}"
    shutil.rmtree(f"{src}/sparse_{n_views}", ignore_errors=True)    # InstantSplat writes here; start clean
    shutil.rmtree(model, ignore_errors=True)
    os.makedirs(model, exist_ok=True)
    os.makedirs(drive_dir, exist_ok=True)

    steps = [
        ("init_geo", f"python -W ignore init_geo.py -s {src} -m {model} --n_views {n_views} --focal_avg --co_vis_dsp --conf_aware_ranking"),
        ("train", f"python train.py -s {src} -m {model} -r 1 --n_views {n_views} --iterations {ITERS} --pp_optimizer --optim_pose"),
        ("render_test", f"python render.py -s {src} -m {model} -r 1 --n_views {n_views} --iterations {ITERS} --eval --optim_test_pose_iter {OPTIM_TEST_POSE_ITER}"),
        ("metrics", f"python metrics.py -s {src} -m {model} --n_views {n_views}"),
    ]
    sampler = VramSampler()
    sampler.start()
    stage_s, t_all = {}, time.time()
    print(f"=== {run_name}")
    for name, cmd in steps:
        t0 = time.time()
        log = f"{model}/{name}.log"
        with open(log, "w") as lf:
            rc = subprocess.run(cmd, shell=True, cwd=REPO, stdout=lf, stderr=subprocess.STDOUT).returncode
        stage_s[name] = round(time.time() - t0, 1)
        print(f"   {name:12s} {stage_s[name]:7.1f} s")
        if rc != 0:
            sampler.stop_flag = True
            tail = "".join(open(log).readlines()[-25:])
            raise RuntimeError(f"Stage '{name}' failed for {run_name}. Last lines of the log:\n{tail}")
    sampler.stop_flag = True

    rj = f"{model}/results.json"
    if not os.path.exists(rj):
        tail = "".join(open(f"{model}/metrics.log").readlines()[-25:])
        raise RuntimeError(f"metrics.py did not write results.json for {run_name}. Log:\n{tail}")
    res = json.load(open(rj))
    m = res[sorted(res)[0]]
    renders = Path(f"{model}/test/ours_{ITERS}/renders")

    # pose error with OUR script: training cameras, before (pose_org) and after (pose_optimized) refinement
    pose_dir = Path(f"{model}/pose/ours_{ITERS}")
    names = manifest[scene]["train_images"][str(n_views)]
    gt_bin = f"{src}/sparse/0/images.bin"
    pe_after = pose_error.evaluate_run(gt_bin, pose_dir / "pose_optimized.npy", names)
    pe_before = pose_error.evaluate_run(gt_bin, pose_dir / "pose_org.npy", names)
    os.makedirs(f"{drive_dir}/poses", exist_ok=True)
    for f in ["pose_org.npy", "pose_optimized.npy"]:
        shutil.copy(pose_dir / f, f"{drive_dir}/poses/{f}")

    result = dict(config=tag, scene=scene, n_views=n_views,
                  psnr=m["PSNR"], ssim=m["SSIM"], lpips=m["LPIPS"],
                  **{k: pe_after[k] for k in POSE_KEYS},
                  **{k + "_init": pe_before[k] for k in POSE_KEYS},
                  pose_metric="pose_error.py", train_images=names,
                  instantsplat_pose_metrics_unreliable={"ATE": m.get("ATE"), "RPE_t": m.get("RPE_t"), "RPE_r": m.get("RPE_r")},
                  n_test_images=len(list(renders.glob("*"))),
                  stage_seconds=stage_s, total_seconds=round(time.time() - t_all, 1),
                  peak_vram_mb=sampler.peak, gpu=GPU_NAME, resolution=RES_DIR, iterations=ITERS,
                  optim_test_pose_iter=OPTIM_TEST_POSE_ITER, test_stride=TEST_STRIDE,
                  instantsplat_commit=COMMIT, date=datetime.datetime.now().isoformat(timespec="seconds"))

    # keep the small files on Drive (logs, metrics, test renders); the heavy model stays local
    for f in ["results.json", "per_view.json", "train_time.txt"] + [p.name for p in Path(model).glob("*.log")]:
        if os.path.exists(f"{model}/{f}"):
            shutil.copy(f"{model}/{f}", drive_dir)
    shutil.copytree(renders, f"{drive_dir}/test_renders", dirs_exist_ok=True)
    json.dump(result, open(result_file, "w"), indent=1)
    print(f"   done: PSNR {result['psnr']:.2f}  SSIM {result['ssim']:.3f}  LPIPS {result['lpips']:.3f}  | "
          f"pose before->after refinement: ATE {result['ate_init']:.3f}->{result['ate']:.3f}, "
          f"rot {result['rot_deg_init']:.2f}->{result['rot_deg']:.2f} deg  | "
          f"{result['total_seconds']/60:.1f} min | peak GPU {result['peak_vram_mb']} MB")
    return result


print("OK - run_one() is ready")

## Step 7 - Baseline: full InstantSplat on 2 scenes
**What we're doing:** running the full InstantSplat (our configuration **C3**) on `garden` and `bonsai` with 3, 6 and 9 training photos = **6 runs**, one after another.
**Why it matters:** this is milestone M1. If our numbers are close to the published ones, our setup is correct and we can trust the new experiments. The time per run also tells us if the 96-run batch is realistic.
If a run fails, this cell stops and shows the end of the log - copy it to Farida.

In [ ]:
t_start = time.time()
baseline = []
for scene in SCENES_BASELINE:
    for v in VIEWS:
        baseline.append(run_one(scene, v))
print(f"\nOK - {len(baseline)} runs ready. Total time this session: {(time.time() - t_start) / 60:.1f} min")

## Step 8 - Results and text for LOG.md
**What we're doing:** collecting the 6 results into one table, comparing them with the Free360 numbers, and printing a short text to paste into `LOG.md`.
**Why it matters:** this is the evidence for milestone M1.

**Be careful:** the Free360 numbers are **averages over 8 scenes**. We only ran 2 scenes, so small differences are normal. A good sign is: our numbers are in a similar range **and get better as the number of photos grows**.

In [ ]:
import pandas as pd

rows = [json.load(open(p)) for p in sorted(Path(f"{DRIVE}/results/{TAG}").glob("*/*/result.json"))]
df = pd.DataFrame(rows)
cols = ["scene", "n_views", "psnr", "ssim", "lpips", "ate_init", "ate", "rot_deg_init", "rot_deg", "rpe_deg",
        "n_test_images", "total_seconds", "peak_vram_mb"]
table = df[cols].sort_values(["scene", "n_views"]).round(3)
display(table)

avg = df.groupby("n_views")[["psnr", "ssim", "lpips"]].mean().round(3)
target = pd.DataFrame({"psnr": [14.10, 15.83, 17.16], "ssim": [0.296, 0.351, 0.402], "lpips": [0.529, 0.480, 0.443]},
                      index=pd.Index([3, 6, 9], name="n_views"))
cmp_ = avg.join(target, rsuffix="_free360")
cmp_["psnr_diff"] = (cmp_["psnr"] - cmp_["psnr_free360"]).round(2)
print("\nAverage over our scenes vs Free360 (8-scene average):")
display(cmp_)

table.to_csv(f"{DRIVE}/results/{TAG}_summary.csv", index=False)
mean_min = df["total_seconds"].mean() / 60
print(f"\nMean time per run: {mean_min:.1f} min  ->  96 runs would take about {96 * mean_min / 60:.1f} GPU-hours "
      f"(about {48 * mean_min / 60:.1f} h per account).")

log = f'''### LOG entry - Block 1 baseline ({datetime.date.today()})
- GPU: {GPU_NAME} | InstantSplat commit: {COMMIT} | image size: {RES_DIR} | iterations: {ITERS}
- Split rule: test = every {TEST_STRIDE}th photo; train = n photos spaced evenly among the rest (Free360-style)
- Scenes: {", ".join(SCENES_BASELINE)} | photos: {VIEWS} | config: {TAG}
- Average PSNR over our scenes: ''' + ", ".join(f"{v} views = {avg.loc[v, 'psnr']:.2f}" for v in avg.index) + f'''
- Free360 PSNR (8-scene average): 3 views = 14.10, 6 views = 15.83, 9 views = 17.16
- Mean time per run: {mean_min:.1f} min | peak GPU memory: {int(df["peak_vram_mb"].max())} MB
- Pose error (our pose_error.py, training cameras, mean over scenes) before -> after refinement: ''' + ", ".join(
    f"{v} views: ATE {df[df.n_views == v].ate_init.mean():.3f} -> {df[df.n_views == v].ate.mean():.3f}, "
    f"rot {df[df.n_views == v].rot_deg_init.mean():.2f} -> {df[df.n_views == v].rot_deg.mean():.2f} deg" for v in avg.index) + f'''
- Decision on M1: {"PASSED - every view count within 1.5 dB of Free360" if (cmp_["psnr_diff"].abs() < 1.5).all() else "NOT within 1.5 dB - document why and freeze the protocol"}
'''
print("\n" + log)

## Step 9 - What is saved and what is next
**Saved on Drive (`3dgs-project/`):**
- `setup_colab.sh`, `wheels/`, `checkpoints/`, `requirements-colab.txt` - restore the setup in a new session (just run this notebook again)
- `views_manifest.json` - the exact training/test photos of every scene
- `data/mipnerf360_8scenes_images_8.tar` - the dataset copy (photos + reference poses)
- `scripts/pose_error.py` - our pose-error script
- `results/C3_baseline/<scene>/<n>_views/` - result.json, logs, metrics, test renders, `poses/` (before/after refinement)
- `results/C3_baseline_summary.csv` - the table above

**Still to do by hand (tracker task F3.2):**
1. Paste the LOG text above into `LOG.md` in the repo.
2. Share the Drive folder with Mona (Editor) if not done yet.
3. Set up Overleaf + Zotero; try to get the DPR-Splat PDF.
4. Hand-off call with Mona (tracker row H1): Mona runs this notebook once on her own account.